# MMSD data inspection


In [3]:
import ast
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

DATA_ROOT = Path('../data')
IMAGE_DIR = DATA_ROOT / 'raw/dataset_image'
assert (DATA_ROOT / 'raw/mmsd/text').is_dir()
assert IMAGE_DIR.is_dir()


In [4]:
blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour',
           'humor', 'jokes', 'irony', 'ironic', 'exgag'}
records = []
invalid_records = []
for split in ('train', 'valid2', 'test2'):
    lines = (DATA_ROOT / f'raw/mmsd/text/{split}.txt').read_text().splitlines()
    valid_count = excluded_count = 0
    for line_number, line in enumerate(lines, 1):
        try:
            row = ast.literal_eval(line)
            if not isinstance(row, (list, tuple)) or len(row) != (3 if split == 'train' else 4):
                raise ValueError('Unexpected record structure')
            if isinstance(row[0], bool) or not isinstance(row[0], (int, str)) or not str(row[0]).isdigit():
                raise ValueError('Invalid tweet ID')
            if not isinstance(row[1], str) or not row[1].strip():
                raise ValueError('Empty or invalid text')
            if any(type(label) is not int or label not in (0, 1) for label in row[2:]):
                raise ValueError('Invalid binary label')
            excluded = bool(blocked.intersection(row[1].split()))
            records.append((split, line_number, str(row[0]), ' '.join(row[1].split()), row[-1], excluded))
            valid_count += 1
            excluded_count += excluded
        except (ValueError, SyntaxError, TypeError) as error:
            invalid_records.append((split, line_number, str(error)))
    print(split, {'rows': len(lines), 'valid_records': valid_count,
                  'excluded_by_authors_filter': excluded_count, 'retained': valid_count - excluded_count})
print('Invalid records:', len(invalid_records), invalid_records[:5])
for name, inspected in [('raw', records), ('filtered', [r for r in records if not r[5]])]:
    print(name.upper())
    for field, index in [('tweet_id', 2), ('text', 3)]:
        groups = {}
        for record in inspected:
            groups.setdefault(record[index], []).append((record[0], record[1], record[4]))
        duplicates = {key: locations for key, locations in groups.items() if len(locations) > 1}
        cross_split = {key: locations for key, locations in duplicates.items()
                       if len({location[0] for location in locations}) > 1}
        print(field, {'duplicate_groups': len(duplicates),
                      'extra_rows': sum(len(v) - 1 for v in duplicates.values()),
                      'cross_split_groups': len(cross_split)})
        print('Examples (split, line, final label):', list(duplicates.items())[:5])
        print('Cross-split examples:', list(cross_split.items())[:5])
image_paths = {path.stem: path for path in IMAGE_DIR.glob('*.jpg')}
raw_ids = {r[2] for r in records}
retained_ids = {r[2] for r in records if not r[5]}
for name, ids in [('raw', raw_ids), ('filtered', retained_ids)]:
    missing = sorted(ids - image_paths.keys())
    print(name, 'missing images:', len(missing), missing[:5])
extra_images = sorted(image_paths.keys() - retained_ids)
print('Images outside filtered records:', len(extra_images), extra_images[:5])
invalid_images = []
for tweet_id, path in tqdm(sorted(image_paths.items()), desc='Inspect images', unit='image'):
    try:
        with Image.open(path) as image:
            image.convert('RGB').load()
    except (OSError, ValueError, Image.DecompressionBombError) as error:
        invalid_images.append((tweet_id, str(error)))
print('Unreadable images:', len(invalid_images), invalid_images[:5])


train {'rows': 29040, 'valid_records': 29040, 'excluded_by_authors_filter': 9224, 'retained': 19816}
valid2 {'rows': 2410, 'valid_records': 2410, 'excluded_by_authors_filter': 0, 'retained': 2410}
test2 {'rows': 2409, 'valid_records': 2409, 'excluded_by_authors_filter': 0, 'retained': 2409}
Invalid records: 0 []
RAW
tweet_id {'duplicate_groups': 0, 'extra_rows': 0, 'cross_split_groups': 0}
Examples (split, line, final label): []
Cross-split examples: []
text {'duplicate_groups': 548, 'extra_rows': 1794, 'cross_split_groups': 73}
Examples (split, line, final label): [('most # funny quotes : 21 snarky and # funny quotes # # funnyquotes # hilariousquotes # humor # lol ... <url> …', [('train', 1, 1), ('train', 317, 1), ('train', 1184, 1), ('train', 1720, 1), ('train', 5330, 1), ('train', 12075, 1), ('train', 15846, 1), ('train', 19389, 1), ('train', 22045, 1), ('train', 23521, 1), ('train', 26927, 1)]), ('22 super funny quotes # funnyquotes # funnysayings # hilariousquotes # quotes <user>'

Inspect images:   0%|          | 0/24635 [00:00<?, ?image/s]

Unreadable images: 0 []
